# 📧 Spam Email Detection using Scikit-Learn

**Internship Task – 4 | Machine Learning Model Implementation**

| Field | Details |
|---|---|
| **Name** | Ankush Dhanokar |
| **Intern ID** | CTIS9672 |
| **Company** | Codtech IT Solutions |
| **Domain** | Machine Learning |
| **Duration** | 12 Weeks |
| **Mentor** | Neela Santosh |

---

## Objective
Build a machine learning classifier to detect **Spam** vs **Ham (Not Spam)** emails using:
- **TF-IDF Vectorization** for feature extraction
- **Multinomial Naive Bayes** classifier
- Full evaluation with accuracy, precision, recall, F1-score & confusion matrix

---
## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

print('All libraries imported successfully!')

---
## Step 2: Load Dataset

The dataset contains labeled email messages:
- **spam** — Unwanted/junk messages
- **ham** — Legitimate messages

In [ ]:
data = pd.read_csv('spam.csv')

print(f'Dataset Shape: {data.shape}')
print(f'Rows: {data.shape[0]} | Columns: {data.shape[1]}')
print('\nLabel counts:')
print(data['label'].value_counts())
print('\nFirst 5 rows:')
data.head()

In [ ]:
# Check for missing values
print('Missing Values:')
print(data.isnull().sum())

---
## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
data['msg_length'] = data['message'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Exploratory Data Analysis', fontsize=15, fontweight='bold')

# Class distribution
counts = data['label'].value_counts()
colors = ['#4CAF50', '#F44336']
axes[0].bar(counts.index, counts.values, color=colors, edgecolor='black', alpha=0.85, width=0.5)
axes[0].set_title('Class Distribution (Ham vs Spam)')
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 0.5, str(val), ha='center', fontweight='bold', fontsize=13)

# Message length
axes[1].hist(data[data['label']=='ham']['msg_length'],  bins=20, alpha=0.7, label='Ham',  color='#4CAF50')
axes[1].hist(data[data['label']=='spam']['msg_length'], bins=20, alpha=0.7, label='Spam', color='#F44336')
axes[1].set_title('Message Length Distribution')
axes[1].set_xlabel('Character Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Avg spam message length : {data[data["label"]=="spam"]["msg_length"].mean():.0f} chars')
print(f'Avg ham  message length : {data[data["label"]=="ham"]["msg_length"].mean():.0f} chars')

---
## Step 4: Data Preprocessing

Convert text labels to numerical values:
- **ham → 0**
- **spam → 1**

In [ ]:
data['label_num'] = data['label'].map({'ham': 0, 'spam': 1})

print('Label encoding applied:')
print(data[['label', 'label_num']].drop_duplicates())
data[['label', 'label_num', 'message']].head()

---
## Step 5: Train-Test Split

In [ ]:
X = data['message']    # Features
y = data['label_num']  # Labels

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Total samples  : {len(data)}')
print(f'Training set   : {len(X_train)} samples (80%)')
print(f'Test set       : {len(X_test)} samples (20%)')
print(f'\nTrain label distribution:')
print(y_train.value_counts().rename({0: 'ham', 1: 'spam'}))

---
## Step 6: Feature Extraction — TF-IDF Vectorization

**TF-IDF** converts raw text into a matrix of numerical features.
- **TF**: How often a word appears in a message
- **IDF**: Penalizes words that appear in almost all messages
- `stop_words='english'` removes common words like 'the', 'is', 'and'

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=500)

# Fit ONLY on training data — transform both sets
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print(f'Vocabulary size : {len(vectorizer.vocabulary_)} words')
print(f'Training matrix : {X_train_tfidf.shape}')
print(f'Test matrix     : {X_test_tfidf.shape}')
print(f'\nSample features: {list(vectorizer.get_feature_names_out()[:20])}')

---
## Step 7: Train the Model — Multinomial Naive Bayes

**Why Naive Bayes for spam detection?**
- Works extremely well with text/word frequency data
- Fast to train, even on large datasets
- Based on: P(Spam | words in email)

In [ ]:
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

print('Model trained successfully!')
print(f'Algorithm : Multinomial Naive Bayes')
print(f'Classes   : {model.classes_}  ->  [0=Ham, 1=Spam]')

---
## Step 8: Make Predictions

In [ ]:
y_pred      = model.predict(X_test_tfidf)
y_pred_prob = model.predict_proba(X_test_tfidf)

results = pd.DataFrame({
    'Message'    : X_test.values,
    'Actual'     : y_test.map({0: 'Ham', 1: 'Spam'}).values,
    'Predicted'  : pd.Series(y_pred).map({0: 'Ham', 1: 'Spam'}).values,
    'Spam Prob %': (y_pred_prob[:, 1] * 100).round(1)
})
results['Correct'] = results['Actual'] == results['Predicted']
print('Sample predictions (first 10):')
results.head(10)

---
## Step 9: Model Evaluation

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print(f'{'='*45}')
print(f'  MODEL ACCURACY : {accuracy*100:.2f}%')
print(f'{'='*45}')
print()
print('Detailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Ham (0)', 'Spam (1)']))
print('Metrics Explained:')
print('  Precision -> Of emails predicted Spam, how many were actually spam?')
print('  Recall    -> Of all actual Spam emails, how many did we catch?')
print('  F1-Score  -> Harmonic mean of Precision and Recall')

---
## Step 10: Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'],
            ax=ax, linewidths=0.5, linecolor='gray',
            annot_kws={'size': 16})
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('Actual Label', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'True Negatives  (Ham correctly identified)  : {tn}')
print(f'False Positives (Ham wrongly flagged spam)  : {fp}')
print(f'False Negatives (Spam missed by model)      : {fn}')
print(f'True Positives  (Spam correctly caught)     : {tp}')

---
## Step 11: Top Spam Indicator Words

In [ ]:
feature_names   = np.array(vectorizer.get_feature_names_out())
spam_log_probs  = model.feature_log_prob_[1]
top_idx         = spam_log_probs.argsort()[-20:][::-1]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(feature_names[top_idx][::-1], spam_log_probs[top_idx][::-1],
        color='#F44336', alpha=0.8, edgecolor='black', linewidth=0.5)
ax.set_title('Top 20 Spam Indicator Words (Log Probabilities)', fontsize=12, fontweight='bold')
ax.set_xlabel('Log Probability')
plt.tight_layout()
plt.show()

---
## Step 12: Test Custom Emails

> **Try it yourself!** Edit the emails in `test_emails` below.

In [ ]:
# --- Edit these emails to test your own messages ---
test_emails = [
    'Congratulations! You have won a free iPhone. Claim your prize now!',
    'Hi, the meeting is rescheduled to 3pm tomorrow.',
    'URGENT: Click here to verify your bank account immediately.',
    'Can you bring lunch tomorrow? I will bring dessert.',
    'You are selected to receive a Rs 5000 gift card. Reply now!',
    'Don not forget to submit the report by Friday.',
]
# ---------------------------------------------------

print('=' * 65)
print('  SPAM DETECTOR - Custom Email Predictions')
print('=' * 65)

for email in test_emails:
    vec  = vectorizer.transform([email])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0]
    label = 'SPAM' if pred == 1 else 'HAM '
    bar   = '#' * int(prob[1] * 20)
    print(f'\n[{label}] | Spam confidence: {prob[1]*100:5.1f}% |{bar}')
    print(f'       -> {email[:72]}')

---
## Summary & Conclusion

| Step | Task | Tool Used |
|------|------|-----------|
| 1 | Data Loading & Exploration | Pandas |
| 2 | Label Encoding | Pandas `.map()` |
| 3 | Train/Test Split (80/20) | `train_test_split` |
| 4 | Text Vectorization | `TfidfVectorizer` |
| 5 | Model Training | `MultinomialNB` |
| 6 | Evaluation | `classification_report`, `confusion_matrix` |
| 7 | Visualization | Matplotlib, Seaborn |

### Key Takeaways
- **TF-IDF** effectively captured spam signal words like 'free', 'win', 'urgent', 'claim'
- **Multinomial Naive Bayes** is highly suitable for text classification tasks
- The model achieved strong accuracy and can be further improved with a larger real-world dataset

### Future Improvements
- Train on the full SMS Spam Collection dataset (5,574 messages)
- Try other models: Logistic Regression, SVM, Random Forest
- Add bigrams/trigrams to the TF-IDF vectorizer
- Deploy as a web app using Flask or Streamlit

---
*Internship Task-4 | Codtech IT Solutions | Intern: Ankush Dhanokar (CTIS9672)*